In [1]:
!pip install grpcio grpcio-tools click

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 14.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.64.1
    Uninstalling grpcio-1.64.1:
      Successfully uninstalled grpcio-1.64.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires protobuf<5,>=3.20, but you have protobuf 5.28.2 which is incompatible.
google-ai-generativelanguage 0.6.6 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.28.2 wh

In [ ]:
%%writefile remote_control.proto
syntax = "proto3";

package remote_control;

service RemoteControl {
    rpc ExecuteCommand (CommandRequest) returns (CommandResponse);
}

message CommandRequest {
    string command = 1;
}

message CommandResponse {
    string result = 1;
    int32 status_code = 2;
}


Writing remote_control.proto


In [ ]:
!python -m grpc_tools.protoc -I./ --python_out=. --grpc_python_out=. remote_control.proto


In [ ]:
import grpc
from concurrent import futures
import subprocess
import remote_control_pb2_grpc
import remote_control_pb2

class RemoteControlServicer(remote_control_pb2_grpc.RemoteControlServicer):
    def ExecuteCommand(self, request, context):
        try:
            result = subprocess.run(request.command, shell=True, capture_output=True, text=True)
            return remote_control_pb2.CommandResponse(result=result.stdout, status_code=result.returncode)
        except Exception as e:
            return remote_control_pb2.CommandResponse(result=str(e), status_code=-1)

def serve():
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
    remote_control_pb2_grpc.add_RemoteControlServicer_to_server(RemoteControlServicer(), server)
    server.add_insecure_port('[::]:50051')
    server.start()
    print("Servidor gRPC iniciado na porta 50051")
    server.wait_for_termination()

# Execute o servidor em segundo plano
import threading
server_thread = threading.Thread(target=serve)
server_thread.start()


Servidor gRPC iniciado na porta 50051


In [ ]:
import grpc
import remote_control_pb2_grpc
import remote_control_pb2

def execute_command(command):
    with grpc.insecure_channel('localhost:50051') as channel:
        stub = remote_control_pb2_grpc.RemoteControlStub(channel)
        response = stub.ExecuteCommand(remote_control_pb2.CommandRequest(command=command))
        return response.result, response.status_code

# Teste o cliente enviando um comando
resultado, status_code = execute_command('ls -l')
print(f"Resultado:\n{resultado}")
print(f"Código de Status: {status_code}")


Resultado:
total 20
drwxr-xr-x 2 root root 4096 Aug 12 18:21 __pycache__
-rw-r--r-- 1 root root 3791 Aug 12 18:21 remote_control_pb2_grpc.py
-rw-r--r-- 1 root root 1514 Aug 12 18:21 remote_control_pb2.py
-rw-r--r-- 1 root root  269 Aug 12 18:20 remote_control.proto
drwxr-xr-x 1 root root 4096 Aug  9 13:22 sample_data

Código de Status: 0


In [ ]:
# Enviar um comando para listar os arquivos no diretório atual
resultado, status_code = execute_command('ls -l')
print(f"Resultado:\n{resultado}")
print(f"Código de Status: {status_code}")



Resultado:
total 20
drwxr-xr-x 2 root root 4096 Aug 12 18:21 __pycache__
-rw-r--r-- 1 root root 3791 Aug 12 18:21 remote_control_pb2_grpc.py
-rw-r--r-- 1 root root 1514 Aug 12 18:21 remote_control_pb2.py
-rw-r--r-- 1 root root  269 Aug 12 18:20 remote_control.proto
drwxr-xr-x 1 root root 4096 Aug  9 13:22 sample_data

Código de Status: 0


In [ ]:
# Enviar um comando para verificar o espaço em disco
resultado, status_code = execute_command('df -h')
print(f"Resultado:\n{resultado}")
print(f"Código de Status: {status_code}")


Resultado:
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   31G   77G  29% /
tmpfs            64M     0   64M   0% /dev
shm             5.8G     0  5.8G   0% /dev/shm
/dev/root       2.0G  1.2G  820M  59% /usr/sbin/docker-init
tmpfs           6.4G  808K  6.4G   1% /var/colab
/dev/sda1        70G   51G   20G  72% /kaggle/input
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware

Código de Status: 0
